# 🔬 ضبط TrOCR للملاحظات الطبية العربية — TrOCR Fine-Tuner

دفتر تدريب مخصص لنموذج TrOCR على بيانات الملاحظات الطبية المكتوبة بخط اليد.

### الميزات:
- تحميل بيانات التدريب (images/ + labels.txt)
- ضبط microsoft/trocr-base-handwritten
- قياس الأداء بـ CER (Character Error Rate)
- تصدير النموذج كملف ZIP للاستخدام بدون إنترنت

### الإعدادات:
- EPOCHS = 50
- BATCH_SIZE = 4
- LEARNING_RATE = 5e-5

## الخطوة ١: تثبيت الحزم اللازمة

In [ ]:
# تثبيت حزم التدريب والمعالجة
!pip install -q transformers datasets accelerate jiwer pillow opencv-python-headless

# التحقق من التثبيت
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from datasets import Dataset
import jiwer

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ GPU متاح: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ اسم GPU: {torch.cuda.get_device_name(0)}")

## الخطوة ٢: إعداد الإعدادات ومسارات البيانات

In [ ]:
import os
import gc
import json
import zipfile
import numpy as np
from pathlib import Path
from PIL import Image
from typing import List, Dict, Tuple
from dataclasses import dataclass
from datetime import datetime

import torch
from torch.utils.data import Dataset as TorchDataset


# ══════════════════════════════════════════════════════════════
# إعدادات التدريب الرئيسية
# ══════════════════════════════════════════════════════════════

@dataclass
class TrainConfig:
    """إعدادات التدريب للنموذج."""
    # اسم النموذج الأساسي
    model_name: str = "microsoft/trocr-base-handwritten"
    # مجلد حفظ النتائج
    output_dir: str = "./trocr_medical_finetuned"
    # مجلد بيانات التدريب
    data_dir: str = "./dataset"
    # عدد حقب التدريب
    epochs: int = 50
    # حجم الدفعة
    batch_size: int = 4
    # معدّل التعلّم
    learning_rate: float = 5e-5
    # أقصى طول للنص (حرفًا)
    max_length: int = 128
    # نسبة بيانات التدريب (الباقي للاختبار)
    train_split: float = 0.9
    # بذرة عشوائية
    seed: int = 42


# إنشاء كائن الإعدادات
config = TrainConfig()

print("=" * 50)
print("⚙️  إعدادات التدريب:")
print(f"  النموذج:  {config.model_name}")
print(f"  الحقب:     {config.epochs}")
print(f"  الدفعة:    {config.batch_size}")
print(f"  التعلّم:   {config.learning_rate}")
print(f"  البيانات:  {config.data_dir}")
print("=" * 50)

## الخطوة ٣: تحميل البيانات من المجلد (images/ + labels.txt)

In [ ]:
class MedicalOCRDataset(TorchDataset):
    """مجموعة بيانات OCR الطبية للخط اليدوي.

    يقرأ البيانات من:
        data_dir/
        ├── images/
        │   ├── line_0001.png
        │   └── ...
        └── labels.txt   (تنسيق: اسم_الصورة\tالنص)
    """

    def __init__(
        self,
        data_dir: str,
        processor: TrOCRProcessor,
        max_length: int = 128,
    ):
        self.processor = processor
        self.max_length = max_length
        self.samples: List[Dict[str, str]] = []

        # مسارات الملفات
        data_path = Path(data_dir)
        images_dir = data_path / "images"
        labels_file = data_path / "labels.txt"

        # التحقق من وجود الملفات
        if not labels_file.exists():
            raise FileNotFoundError(
                f"لم يُعثر على ملف labels.txt في: {data_dir}"
            )
        if not images_dir.exists():
            raise FileNotFoundError(
                f"لم يُعثر على مجلد images/ في: {data_dir}"
            )

        # ── قراءة ملف التصنيفات ──
        with open(labels_file, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue  # تخطي الأسطر الفارغة

                # فصل اسم الصورة عن النص
                parts = line.split('\t', 1)
                if len(parts) != 2:
                    print(f"⚠️  تخطي سطر {line_num}: تنسيق غير صالح")
                    continue

                img_name, text = parts[0].strip(), parts[1].strip()
                img_path = images_dir / img_name

                # التحقق من وجود الصورة
                if not img_path.exists():
                    print(f"⚠️  تخطي: الصورة غير موجودة {img_path}")
                    continue

                self.samples.append({
                    "image_path": str(img_path),
                    "text": text,
                })

        print(f"✅ تمّ تحميل {len(self.samples)} عينة من {data_dir}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        sample = self.samples[idx]

        # تحميل الصورة
        image = Image.open(sample["image_path"]).convert("RGB")

        # اقتصاص النص إذا كان طويلًا
        text = sample["text"][:self.max_length]

        # معالجة الصورة بـ TrOCRProcessor
        pixel_values = self.processor(
            image, return_tensors="pt"
        ).pixel_values.squeeze(0)

        # ترميز النص
        labels = self.processor.tokenizer(
            text,
            return_tensors="pt",
            padding="max_length",
            max_length=self.max_length,
            truncation=True,
        ).input_ids.squeeze(0)

        # استبدال pad_token_id بـ -100 (يتجاهلها دالة الخسارة)
        labels[
            labels == self.processor.tokenizer.pad_token_id
        ] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels,
        }


# ══════════════════════════════════════════════════════════════
# توليد بيانات تجريبية (إذا لم تكن بيانات حقيقية متوفرة)
# ══════════════════════════════════════════════════════════════

def generate_synthetic_dataset(output_dir: str, num_samples: int = 300):
    """توليد بيانات تجريبية للملاحظات الطبية.

    Args:
        output_dir: مجلد الإخراج.
        num_samples: عدد العينات.
    """
    import random

    # نصوص طبية عربية تجريبية
    MEDICAL_TEXTS = [
        "المريض يعاني من ارتفاع ضغط الدم",
        "الجرعة: 500 ملغ مرتين يوميا بعد الأكل",
        "أموكسيسيلين 250 ملغ كل 8 ساعات",
        "ميتفورمين 850 ملغ مع الفطور",
        "أنسولين 10 وحدات قبل كل وجبة",
        "بايريكسامول 500 ملغ عند الحاجة",
        "أوميبرازول 20 ملغ قبل النوم",
        "أسبرين 100 ملغ يوميا",
        "ليزينوبريل 10 ملغ صباحا",
        "تشخيص: التهاب الجيوب الأنفية المزمن",
        "المرض: السكري من النوع الثاني",
        "الحالة: ارتفاع ضغط شرياني",
        "صورة دم كاملة ضرورية",
        "تخطيط قلب ECG في الموعد القادم",
        "وظائف الكبد والكلى سليمة",
        "يجب الإقلاع عن التدخين فورا",
        "نظام غذائي قليل الملح والسكر",
        "ممارسة الرياضة 30 دقيقة يوميا",
        "مراجعة بعد شهر للاطمئنان",
        "الوضع مستقر تحت المراقبة",
    ]

    out_path = Path(output_dir)
    images_dir = out_path / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    labels = []
    for i in range(num_samples):
        text = random.choice(MEDICAL_TEXTS)
        # إضافة تنويع عشوائي
        if random.random() < 0.3:
            text = f"المريض رقم {random.randint(100, 999)} — {text}"

        # إنشاء صورة باللون الأبيض
        img = Image.new("RGB", (640, 64), color=(255, 255, 255))
        from PIL import ImageDraw, ImageFont
        draw = ImageDraw.Draw(img)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 24)
        except Exception:
            font = ImageFont.load_default()
        draw.text((10, 15), text, fill=(0, 0, 0), font=font)

        # إضافة ضوضاء خفيفة
        arr = np.array(img)
        noise = np.random.randint(0, 20, arr.shape, dtype=np.uint8)
        arr = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr)

        # حفظ الصورة
        img_name = f"line_{i:04d}.png"
        img.save(str(images_dir / img_name))
        labels.append(f"{img_name}\t{text}")

    # حفظ ملف التصنيفات
    labels_file = out_path / "labels.txt"
    labels_file.write_text("\n".join(labels), encoding="utf-8")
    print(f"✅ تمّ توليد {num_samples} عينة في: {output_dir}")


# ── إنشاء البيانات إذا لم تكن موجودة ──
if not Path(config.data_dir).exists() or not (Path(config.data_dir) / "labels.txt").exists():
    print("📊 لم تُعثر على بيانات — جاري توليد بيانات تجريبية...")
    generate_synthetic_dataset(config.data_dir, num_samples=300)
else:
    print(f"📊 تمّ العثور على بيانات في: {config.data_dir}")

## الخطوة ٤: تحميل النموذج والمعالج وإعداد البيانات

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
from torch.utils.data import random_split

# ── تحديد الجهاز ──
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  الجهاز: {DEVICE}")

# ── تحميل المعالج (Processor) ──
print("⏳ جاري تحميل المعالج TrOCRProcessor...")
processor = TrOCRProcessor.from_pretrained(config.model_name)

# ── تحميل النموذج الأساسي ──
print(f"⏳ جاري تحميل النموذج: {config.model_name}")
model = VisionEncoderDecoderModel.from_pretrained(config.model_name)

# ── تكوين معرّفات خاصة بالنموذج ──
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

# تعيين أقصى طول للتوليد
model.config.max_length = config.max_length
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0

print("✅ تمّ تحميل النموذج والمعالج بنجاح")

# ── إنشاء مجموعة البيانات ──
print("⏳ جاري تحميل بيانات التدريب...")
full_dataset = MedicalOCRDataset(
    data_dir=config.data_dir,
    processor=processor,
    max_length=config.max_length,
)

# ── تقسيم التدريب والاختبار ──
total_size = len(full_dataset)
train_size = int(total_size * config.train_split)
eval_size = total_size - train_size

train_dataset, eval_dataset = random_split(
    full_dataset,
    [train_size, eval_size],
    generator=torch.Generator().manual_seed(config.seed),
)

print(f"📊 التدريب: {len(train_dataset)} عينة")
print(f"📊 الاختبار: {len(eval_dataset)} عينة")

## الخطوة ٥: إعداد التدريب مع مقياس CER

In [ ]:
import jiwer


def compute_metrics(eval_pred):
    """حساب Character Error Rate (CER) لمقارنة التنبؤات بالحقيقة.

    CER = (عمليات الحذف + الإدراج + الاستبدال) / عدد أحرف الحقيقة

    Args:
        eval_pred: كائن يحتوي التنبؤات والتصنيفات.

    Returns:
        قاموس يحتوي قيمة CER.
    """
    predictions_ids = eval_pred.predictions
    labels_ids = eval_pred.label_ids

    # فك ترميز التنبؤات إلى نصوص
    pred_str = processor.tokenizer.batch_decode(
        predictions_ids, skip_special_tokens=True
    )

    # استبدال -100 بمعرّف الحشو قبل فك الترميز
    labels_ids = np.where(
        labels_ids != -100,
        labels_ids,
        processor.tokenizer.pad_token_id,
    )
    label_str = processor.tokenizer.batch_decode(
        labels_ids, skip_special_tokens=True
    )

    # تصفية النصوص الفارغة
    filtered_preds = []
    filtered_labels = []
    for p, l in zip(pred_str, label_str):
        if l.strip():  # تجاهل التصنيفات الفارغة
            filtered_preds.append(p)
            filtered_labels.append(l)

    if not filtered_labels:
        return {"cer": 0.0}

    # حساب CER
    cer_score = jiwer.cer(filtered_labels, filtered_preds)
    return {"cer": cer_score}


print("✅ تمّ تعريف دالة تقييم CER")

## الخطوة ٦: تشغيل التدريب

In [ ]:
import time

start_time = time.time()

# ── إعداد مُعامِلات التدريب ──
training_args = Seq2SeqTrainingArguments(
    output_dir=config.output_dir,
    # إعدادات الحقب والدفعة
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    # إعدادات التعلّم
    learning_rate=config.learning_rate,
    warmup_ratio=0.1,
    weight_decay=0.01,
    # التنبؤ والتقييم
    predict_with_generate=True,
    generation_max_length=config.max_length,
    fp16=(DEVICE == "cuda"),  # تدريب بنصف دقة على GPU
    # استراتيجية الحفظ والتقييم
    save_strategy="epoch",
    evaluation_strategy="epoch",
    save_total_limit=3,       # الاحتفاظ بآخر 3 نقاط حفظ
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,  # CER كلما قلّ كان أفضل
    # التسجيل
    logging_steps=10,
    logging_dir=f"{config.output_dir}/logs",
    report_to="none",
    # تحسين الذاكرة
    gradient_accumulation_steps=2,
    dataloader_pin_memory=(DEVICE == "cuda"),
    # إيقاف مبكر (إذا لم يتحسن CER لمدة 5 حقب)
)

print(f"⚙️  إعدادات التدريب:")
print(f"  الحقب:          {config.epochs}")
print(f"  حجم الدفعة:     {config.batch_size}")
print(f"  معدّل التعلّم:  {config.learning_rate}")
print(f"  نصف دقة (fp16): {DEVICE == 'cuda'}")

# ── إنشاء المدرب (Trainer) ──
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=processor.tokenizer,
    compute_metrics=compute_metrics,
)

# ── بدء التدريب ──
print(f"\n🏋️  بدء التدريب — {config.epochs} حقب...")
print("=" * 50)

train_result = trainer.train()

elapsed = time.time() - start_time
print(f"\n✅ تمّ التدريب في {elapsed:.1f} ثانية")

# ── حفظ النموذج النهائي ──
final_dir = os.path.join(config.output_dir, "final_model")
trainer.save_model(final_dir)
processor.save_pretrained(final_dir)

# ── حفظ إحصائيات التدريب ──
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

# ── تقييم النهائي ──
print("\n📊 جاري التقييم النهائي...")
eval_results = trainer.evaluate()
trainer.log_metrics("eval", eval_results)
trainer.save_metrics("eval", eval_results)

print(f"\n📈 CER النهائي: {eval_results.get('eval_cer', 'N/A'):.4f}")
print(f"📁 النموذج محفوظ في: {final_dir}")

## الخطوة ٧: تصدير النموذج كملف ZIP

In [ ]:
def export_model_as_zip(
    model_dir: str,
    output_zip: str = "/content/trocr_medical_model.zip",
) -> str:
    """تصدير النموذج المُدرب كملف ZIP للاستخدام بدون إنترنت.

    يحتوي الملف على:
        - config.json: إعدادات النموذج
        - model.safetensors: أوزان النموذج
        - tokenizer_config.json + المفردات: محرّك النصوص
        - preprocessor_config.json: إعدادات المعالج
        - training_config.json: معلومات التدريب

    Args:
        model_dir: مسار مجلد النموذج.
        output_zip: مسار ملف ZIP الناتج.

    Returns:
        مسار ملف ZIP.
    """
    model_path = Path(model_dir)
    if not model_path.exists():
        raise FileNotFoundError(f"مجلد النموذج غير موجود: {model_dir}")

    # ── جمع الملفات ──
    files_to_pack = []
    for pattern in ["*.json", "*.safetensors", "*.bin", "*.txt", "*.model"]:
        files_to_pack.extend(model_path.glob(pattern))

    # ── إنشاء ملف إعدادات التدريب ──
    training_info = {
        "model_name": config.model_name,
        "epochs": config.epochs,
        "batch_size": config.batch_size,
        "learning_rate": config.learning_rate,
        "train_samples": len(train_dataset),
        "eval_samples": len(eval_dataset),
        "final_cer": eval_results.get("eval_cer", None),
        "training_time_seconds": round(elapsed, 2),
        "trained_at": datetime.now().isoformat(),
        "device": DEVICE,
    }
    training_info_path = model_path / "training_config.json"
    training_info_path.write_text(
        json.dumps(training_info, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    files_to_pack.append(training_info_path)

    # ── إنشاء سكربت الاستدلال ──
    inference_code = '''#!/usr/bin/env python3
# سكربت الاستدلال — TrOCR Medical (Offline)
import sys
from PIL import Image
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

MODEL_DIR = "./final_model"

def load():
    processor = TrOCRProcessor.from_pretrained(MODEL_DIR)
    model = VisionEncoderDecoderModel.from_pretrained(MODEL_DIR)
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
    return model, processor

def recognize(image_path, model=None, processor=None):
    if model is None:
        model, processor = load()
    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(image, return_tensors="pt").pixel_values
    if torch.cuda.is_available():
        pixel_values = pixel_values.cuda()
    generated = model.generate(pixel_values)
    text = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    return text

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("الاستخدام: python inference.py <صورة>")
        sys.exit(1)
    print(f"النص: {recognize(sys.argv[1])}")
'''
    inference_path = model_path / "inference.py"
    inference_path.write_text(inference_code, encoding="utf-8")
    files_to_pack.append(inference_path)

    # ── إنشاء ملف ZIP ──
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for file_path in files_to_pack:
            arcname = f"trocr_medical_model/{file_path.name}"
            zf.write(str(file_path), arcname)

    # ── عرض النتائج ──
    zip_size = os.path.getsize(output_zip) / (1024 * 1024)
    print(f"✅ تمّ تصدير النموذج بنجاح!")
    print(f"📦 الملف: {output_zip}")
    print(f"📦 الحجم: {zip_size:.1f} ميجابايت")
    print(f"📊 عدد الملفات: {len(files_to_pack)}")

    return output_zip


# ── تشغيل التصدير ──
zip_path = export_model_as_zip(final_dir)
print(f"\n📁 مسار التنزيل: {zip_path}")

## الخطوة ٨: اختبار سريع للنموذج المُدرب

In [ ]:
# ── اختبار النموذج المُدرب على عينة ──
from PIL import Image
import matplotlib.pyplot as plt

# تحميل النموذج المُدرب
print("⏳ جاري تحميل النموذج المُدرب...")
finetuned_processor = TrOCRProcessor.from_pretrained(final_dir)
finetuned_model = VisionEncoderDecoderModel.from_pretrained(final_dir)
finetuned_model.eval()

if DEVICE == "cuda":
    finetuned_model = finetuned_model.to(DEVICE)

# اختبار على أول 5 صور من مجموعة الاختبار
print("\n📊 اختبار على عينات من مجموعة الاختبار:\n")
print("-" * 60)

test_count = min(5, len(eval_dataset))
for i in range(test_count):
    sample = eval_dataset[i]
    pixel_values = sample["pixel_values"].unsqueeze(0)

    if DEVICE == "cuda":
        pixel_values = pixel_values.to(DEVICE)

    # التنبؤ
    with torch.no_grad():
        generated_ids = finetuned_model.generate(pixel_values)
        pred_text = finetuned_processor.tokenizer.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0]

    # الحقيقة
    labels = sample["labels"]
    labels = np.where(labels != -100, labels, finetuned_processor.tokenizer.pad_token_id)
    true_text = finetuned_processor.tokenizer.decode(
        labels, skip_special_tokens=True
    )

    # حساب CER لهذه العينة
    sample_cer = jiwer.cer(true_text, pred_text) if true_text.strip() else 0.0

    print(f"عينة {i+1}:")
    print(f"  الحقيقة:   {true_text}")
    print(f"  التنبؤ:    {pred_text}")
    print(f"  CER:        {sample_cer:.4f}")
    print("-" * 60)

# ── تنظيف الذاكرة ──
del finetuned_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("\n✅ انتهى الاختبار!")